# Étude descriptive

L'échantillon analysé est `data/sample.csv`, extrait d'un jeu d'événements e-commerce d'environ 50 Go.
Chaque ligne contient `event_time` (UTC), `event_type`, `product_id`, `category_id`, `category_code`, `brand`, `price`, `user_id` et `user_session`.
Les types d'événement sont `view` (page produit), `cart` (ajout au panier) et `purchase` (achat).
La cible métier est l'achat au cours d'une session, afin de proposer une offre pendant le parcours.
Aucun fichier n'est produit.


## Chargement


In [ ]:
import os
import numpy as np
import pandas as pd

data = pd.read_csv(os.path.expanduser("data/sample.csv"))
print("Taille de l'échantillon :", data.shape[0])
print("Taille mémoire : {:2.2f} Mb".format(data.memory_usage().sum() / 1e6))
data.head()


## Temporalité


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.units as munits
import matplotlib.dates as mdates
import datetime

sns.set_theme()

data["event_time"] = pd.to_datetime(data["event_time"])
data["event_day"] = data["event_time"].dt.day
data["event_hour"] = data["event_time"].dt.hour
data["event_minute"] = data["event_time"].dt.minute

# Axe datetime : ConciseDateConverter
converter = mdates.ConciseDateConverter()
munits.registry[np.datetime64] = converter
munits.registry[datetime.date] = converter
munits.registry[datetime.datetime] = converter

print("Date min :", data["event_time"].min())
print("Date max :", data["event_time"].max())

plt.figure(figsize=(14, 9))
sns.histplot(data["event_time"])
plt.xlabel("Heure")
plt.ylabel("Nombre d'événements")
plt.title("Répartition du nombre d'événements le 01-10-2019", fontsize=17)
plt.show()


## Types d'événements


In [ ]:
data["event_type"].unique()


In [ ]:
events_type_counts = data["event_type"].value_counts() / data.shape[0]

fig, ax = plt.subplots(figsize=(12, 8), subplot_kw=dict(aspect="equal"))

wedges, texts = ax.pie(
    events_type_counts,
    wedgeprops=dict(width=0.4),
)

for i, p in enumerate(wedges):
    ang = (p.theta2 - p.theta1) / 2.0 + p.theta1
    y = np.sin(np.deg2rad(ang))
    x = np.cos(np.deg2rad(ang))
    horizontalalignment = {-1: "right", 1: "left"}[int(np.sign(x))]
    ax.annotate(
        "{} : {:2.2f}%".format(events_type_counts.index[i], events_type_counts.iloc[i] * 100),
        xy=(x, y),
        xytext=(1.1 * np.sign(x), 1.4 * y),
        fontsize=16,
        horizontalalignment=horizontalalignment,
    )

ax.set_title("Proportion des types d'événements de l'échantillon", fontsize=18)
plt.show()


## Utilisateurs et sessions

`user_id` identifie un utilisateur. `user_session` identifie une visite, fermée après inactivité.
Plusieurs visites le même jour produisent plusieurs sessions.
`purchased` est recopié sur tous les événements de la session : il vaut 1 si elle contient au moins un achat.


In [ ]:
print("Nombre unique d'utilisateurs :", len(data["user_id"].unique()))
print("Nombre de sessions :", len(data["user_session"].unique()))


In [ ]:
import matplotlib.ticker as mtick

num_sessions_per_user = (
    data[["user_id", "user_session"]]
    .groupby("user_id")
    .count()["user_session"]
    .value_counts()
    .sort_values(ascending=False)
)
num_sessions_per_user /= num_sessions_per_user.sum()

plt.figure(figsize=(14, 9))

n_bars = 12

rects = plt.bar(num_sessions_per_user[:n_bars].index, height=num_sessions_per_user[:n_bars])
for rect in rects:
    height = rect.get_height()
    plt.gca().annotate(
        "{:2.2f}%".format(height * 100),
        xy=(rect.get_x() + rect.get_width() / 2, height),
        xytext=(0, 3),
        textcoords="offset points",
        fontsize=16 * 10 / n_bars,
        ha="center",
        va="bottom",
    )

plt.ylabel("Proportion d'utilisateurs")
plt.gca().yaxis.set_major_formatter(mtick.PercentFormatter(1, 0))
plt.xticks(range(1, n_bars + 1))
plt.xlabel("Nombre de sessions")
plt.title("Proportion d'utilisateurs par nombre de sessions", fontsize=16)
plt.show()


In [ ]:
data_purchase = data.copy()
data_purchase["purchased"] = np.where(data_purchase["event_type"] == "purchase", 1, 0)
# 1 si la session contient au moins un achat, propagé à chaque événement
data_purchase["purchased"] = (
    data_purchase.groupby(["user_session"])["purchased"].transform("max")
)
data_purchase["purchased"].value_counts() / data_purchase["purchased"].shape[0]


## Catégories


In [ ]:
data_categories = data.copy()
data_categories["category"] = data_categories["category_code"].str.split(".", expand=True)[0]

for event_type in ["view", "cart", "purchase"]:
    data_categories["event_{}".format(event_type)] = np.where(
        data_categories["event_type"] == event_type, 1, 0
    )

data_categories = (
    data_categories[["event_view", "event_cart", "event_purchase", "category"]]
    .groupby("category")
    .sum()
)
data_categories.head()


In [ ]:
plt.figure(figsize=(14, 9))

rects = plt.barh(data_categories.index, width=data_categories["event_view"])
for rect in rects:
    width = rect.get_width()
    plt.gca().annotate(
        "{}".format(int(width)),
        xy=(width, rect.get_y()),
        xytext=(30, 5),
        textcoords="offset points",
        fontsize=16 * 10 / n_bars,
        ha="center",
        va="bottom",
    )

plt.ylabel("Catégorie")
plt.xlabel("Nombre de vues")
plt.title("Nombre de vues par catégorie de produit", fontsize=16)
plt.show()


In [ ]:
plt.figure(figsize=(14, 9))

rects = plt.barh(data_categories.index, width=data_categories["event_purchase"])
for rect in rects:
    width = rect.get_width()
    plt.gca().annotate(
        "{}".format(int(width)),
        xy=(width, rect.get_y()),
        xytext=(30, 5),
        textcoords="offset points",
        fontsize=16 * 10 / n_bars,
        ha="center",
        va="bottom",
    )

plt.ylabel("Catégorie")
plt.xlabel("Nombre d'achats")
plt.title("Nombre d'achats par catégorie de produit", fontsize=16)
plt.show()


In [ ]:
plt.figure(figsize=(14, 9))

rects = plt.barh(
    data_categories.index,
    width=data_categories["event_purchase"] / data_categories["event_view"],
)
for rect in rects:
    width = rect.get_width()
    plt.gca().annotate(
        "{:2.2f}%".format(width * 100),
        xy=(width, rect.get_y()),
        xytext=(30, 5),
        textcoords="offset points",
        fontsize=16 * 10 / n_bars,
        ha="center",
        va="bottom",
    )

plt.ylabel("Catégorie")
plt.xlabel("Ratio")
plt.gca().xaxis.set_major_formatter(mtick.PercentFormatter(xmax=1, decimals=1))
plt.title("Ratio du nombre d'achats par nombre de vues", fontsize=16)
plt.show()
